# CorroLLM — Explained Notebook Version

This notebook reorganizes **your original one-cell CorroLLM code** into a cleaner, inspection-oriented workflow.

## What this version does
- keeps the same core logic from your notebook
- explains each block in plain engineering language
- separates setup, analysis, reporting, and interaction into readable sections
- remains suitable for **Google Colab**

## Workflow blocks
1. **Environment Setup**
2. **Image Acquisition**
3. **Preprocessing and Corrosion Analysis**
4. **Annotation / Overlay Generation**
5. **Knowledge Base, Root Cause Analysis, and Summary**
6. **Automated Reporting**
7. **Interactive Run Cell**


## Block 0 — Environment Setup

This block installs the libraries used in your original notebook and imports everything needed for:
- image loading and processing
- corrosion pattern analysis
- annotated image generation
- DOCX and PDF reporting
- interactive file upload in Colab

Your original design is preserved: **no external detection API is required**.


In [ ]:

"""
================================================================================
CORROLLM - SIMPLE COLAB VERSION
Run this notebook in Google Colab
Upload image → Get DOCX/PDF report instantly
NO external tokens needed
================================================================================
"""

# Step 1: Install dependencies
print("📦 Installing dependencies (takes ~2 minutes)...")
import subprocess
import sys

packages = [
    'opencv-python-headless', 'pillow', 'numpy',
    'python-docx', 'reportlab', 'anthropic'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✅ Dependencies installed!\n")

# Step 2: Import everything
import os
import io
import cv2
import json
import time
import base64
import random
import numpy as np
from PIL import Image, ImageDraw
from datetime import datetime
from pathlib import Path

# Document generation
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib import colors

# Anthropic (optional - for enhanced AI analysis)
try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except:
    ANTHROPIC_AVAILABLE = False

# Colab file upload
from google.colab import files
from IPython.display import display, HTML, Image as IPImage

# Create output directory
Path('/content/reports').mkdir(exist_ok=True)

print("✅ Setup complete!\n")


## Block 1 — Image Acquisition

This block corresponds to the **inspection input stage** of CorroLLM.

### Engineering purpose
Bring in raw field images for analysis from assets such as:
- pipelines
- oil and gas facilities
- marine structures
- urban steel infrastructure

In your original notebook, this is done with:
- `files.upload()` for local image upload
- `cv2.imread()` for loading the image into the processing pipeline
- a quick image preview before analysis starts


## Block 2 — Corrosion Analysis Engine

This is the main visual analysis layer from your code.

### What your code does
Your `CorrosionAnalyzer`:
- converts the image to grayscale
- extracts edge information using `cv2.Canny`
- finds candidate corrosion contours
- filters very small regions
- estimates corrosion **type**, **severity**, **confidence**, and **affected area**
- guarantees at least one result so the reporting pipeline can continue

### Why this is useful
This keeps the workflow practical and inspection-friendly. Instead of training a new model, your code uses image statistics and engineering heuristics to produce an interpretable first-pass corrosion assessment.


In [ ]:

# Step 3: AI Analysis Engine
class CorrosionAnalyzer:
    """AI-powered corrosion detection"""

    def __init__(self, anthropic_key=None):
        self.anthropic_key = anthropic_key
        if anthropic_key and ANTHROPIC_AVAILABLE:
            self.claude = Anthropic(api_key=anthropic_key)
            self.use_claude = True
            print("🤖 Claude AI enabled for enhanced analysis!")
        else:
            self.use_claude = False
            print("🤖 Using built-in AI analysis")

    def analyze_image(self, image_array):
        """Analyze image for corrosion"""
        h, w = image_array.shape[:2]
        gray = cv2.cvtColor(image_array, cv2.COLOR_BGR2GRAY)

        # Image statistics
        mean_val = np.mean(gray)
        std_val = np.std(gray)

        # Edge detection for corrosion patterns
        edges = cv2.Canny(gray, 50, 150)
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        detections = []
        min_area = w * h * 0.008  # At least 0.8% of image

        for contour in sorted(contours, key=cv2.contourArea, reverse=True)[:8]:
            area = cv2.contourArea(contour)
            if area < min_area:
                continue

            x, y, cw, ch = cv2.boundingRect(contour)
            roi = gray[y:y+ch, x:x+cw]
            roi_mean = np.mean(roi) if roi.size > 0 else mean_val

            # Classify corrosion
            corr_type = self._classify_type(area, roi_mean, mean_val, w*h)
            severity = self._classify_severity(area/(w*h), roi_mean/255)
            confidence = min(0.95, 0.68 + (area/(w*h)) * 1.8)

            detections.append({
                'type': corr_type,
                'severity': severity,
                'confidence': round(confidence, 2),
                'bbox': [int(x), int(y), int(x+cw), int(y+ch)],
                'area_percent': round((area/(w*h))*100, 2)
            })

        # Ensure at least one detection
        if not detections:
            detections = [{
                'type': 'uniform' if mean_val < 128 else 'pitting',
                'severity': 'moderate',
                'confidence': 0.72,
                'bbox': [w//4, h//4, 3*w//4, 3*h//4],
                'area_percent': 20.0
            }]

        total_area = min(100, sum(d['area_percent'] for d in detections))
        overall_severity = self._get_overall_severity(detections)

        return {
            'detections': detections[:5],
            'overall_severity': overall_severity,
            'total_affected_area': round(total_area, 2)
        }

    def _classify_type(self, area, roi_mean, img_mean, total_area):
        ratio = area / total_area
        if ratio > 0.25:
            return 'uniform'
        elif roi_mean < img_mean * 0.65:
            return 'pitting'
        elif ratio > 0.15:
            return 'erosion'
        else:
            return random.choice(['crevice', 'galvanic', 'stress_cracking'])

    def _classify_severity(self, area_ratio, intensity):
        score = (area_ratio * 0.6) + ((1 - intensity) * 0.4)
        if score > 0.65: return 'critical'
        elif score > 0.45: return 'high'
        elif score > 0.25: return 'moderate'
        else: return 'low'

    def _get_overall_severity(self, detections):
        severity_map = {'low': 1, 'moderate': 2, 'high': 3, 'critical': 4}
        max_sev = max(severity_map[d['severity']] for d in detections)
        return [k for k, v in severity_map.items() if v == max_sev][0]

    def create_overlay(self, image, detections):
        """Create annotated overlay"""
        overlay = image.copy()
        colors_map = {
            'low': (0, 255, 0),
            'moderate': (0, 255, 255),
            'high': (0, 165, 255),
            'critical': (0, 0, 255)
        }

        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            color = colors_map[det['severity']]
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 3)

            label = f"{det['type'].upper()}: {det['severity'].upper()}"
            cv2.rectangle(overlay, (x1, y1-25), (x1+200, y1), color, -1)
            cv2.putText(overlay, label, (x1+5, y1-8),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1)

        # Legend
        h, w = overlay.shape[:2]
        y_pos = 20
        for sev, col in colors_map.items():
            cv2.rectangle(overlay, (w-150, y_pos), (w-120, y_pos+20), col, -1)
            cv2.putText(overlay, sev.upper(), (w-115, y_pos+15),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
            y_pos += 25

        return overlay


## Block 3 — Automated Reporting Layer

This block contains your report generator.

### Engineering purpose
Transform inspection findings into formats that are easy to share with:
- supervisors
- maintenance planners
- integrity engineers
- inspection teams

### Output formats in your code
- **DOCX report** for editable documentation
- **PDF report** for fixed distribution

The report includes:
- timestamp
- material and optional location
- executive summary
- detected corrosion findings
- root cause analysis
- recommendations
- preventive measures


In [ ]:

# Step 4: Report Generator
class ReportGenerator:
    """Generate DOCX and PDF reports"""

    def __init__(self):
        self.output_dir = Path('/content/reports')

    def generate_reports(self, data):
        """Generate both DOCX and PDF"""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

        docx_path = self.output_dir / f'corrosion_report_{timestamp}.docx'
        pdf_path = self.output_dir / f'corrosion_report_{timestamp}.pdf'

        self._create_docx(data, docx_path)
        self._create_pdf(data, pdf_path)

        return str(docx_path), str(pdf_path)

    def _create_docx(self, data, filepath):
        """Create DOCX report"""
        doc = Document()

        # Title
        title = doc.add_heading('CORROSION FAILURE ANALYSIS REPORT', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        doc.add_paragraph(f"Report Generated: {data['timestamp']}")
        doc.add_paragraph(f"Material: {data['material']}")
        if data.get('location'):
            doc.add_paragraph(f"Location: {data['location']}")
        doc.add_paragraph(f"Overall Severity: {data['severity'].upper()}")
        doc.add_paragraph("")

        # Executive Summary
        doc.add_heading('Executive Summary', 1)
        doc.add_paragraph(data['executive_summary'])
        doc.add_paragraph("")

        # Findings
        doc.add_heading('Detailed Findings', 1)
        doc.add_paragraph(f"Total Detections: {len(data['detections'])}")
        doc.add_paragraph(f"Total Affected Area: {data['total_affected_area']:.1f}%")
        doc.add_paragraph("")

        for i, det in enumerate(data['detections'], 1):
            doc.add_paragraph(
                f"{i}. {det['type'].replace('_',' ').title()} - "
                f"Severity: {det['severity'].upper()} - "
                f"Confidence: {det['confidence']:.1%} - "
                f"Area: {det['area_percent']:.1f}%"
            )

        doc.add_paragraph("")

        # Root Cause Analysis
        doc.add_heading('Root Cause Analysis', 1)
        for i, hyp in enumerate(data['rca'], 1):
            doc.add_paragraph(f"\n{i}. {hyp['cause']}")
            doc.add_paragraph(f"Confidence: {hyp['confidence']:.0%}")
            doc.add_paragraph("Evidence:")
            for ev in hyp['evidence']:
                doc.add_paragraph(f"  • {ev}", style='List Bullet')

        doc.add_paragraph("")

        # Recommendations
        doc.add_heading('Recommendations', 1)
        for i, rec in enumerate(data['recommendations'], 1):
            doc.add_paragraph(f"\n{i}. {rec['action']}")
            doc.add_paragraph(f"   Priority: {rec['priority'].upper()}")
            doc.add_paragraph(f"   Cost: {rec['cost']}")
            doc.add_paragraph(f"   Timeline: {rec['timeline']}")

        doc.add_paragraph("")

        # Preventive Measures
        doc.add_heading('Preventive Measures', 1)
        for measure in data['preventive_measures']:
            doc.add_paragraph(f"• {measure}", style='List Bullet')

        doc.save(str(filepath))

    def _create_pdf(self, data, filepath):
        """Create PDF report"""
        doc = SimpleDocTemplate(str(filepath), pagesize=letter)
        story = []
        styles = getSampleStyleSheet()

        # Title
        title_style = ParagraphStyle(
            'Title', parent=styles['Heading1'],
            fontSize=18, textColor=colors.HexColor('#1f4788'),
            spaceAfter=20, alignment=1
        )
        story.append(Paragraph("CORROSION FAILURE ANALYSIS REPORT", title_style))
        story.append(Spacer(1, 0.3*72))

        # Metadata
        story.append(Paragraph(f"<b>Date:</b> {data['timestamp']}", styles['Normal']))
        story.append(Paragraph(f"<b>Material:</b> {data['material']}", styles['Normal']))
        story.append(Paragraph(f"<b>Severity:</b> {data['severity'].upper()}", styles['Normal']))
        story.append(Spacer(1, 0.2*72))

        # Executive Summary
        story.append(Paragraph("<b>Executive Summary</b>", styles['Heading2']))
        story.append(Paragraph(data['executive_summary'], styles['Normal']))
        story.append(Spacer(1, 0.2*72))

        # Findings
        story.append(Paragraph("<b>Detailed Findings</b>", styles['Heading2']))
        for i, det in enumerate(data['detections'], 1):
            text = f"{i}. <b>{det['type'].replace('_',' ').title()}</b> - "
            text += f"Severity: {det['severity'].upper()}, Confidence: {det['confidence']:.1%}"
            story.append(Paragraph(text, styles['Normal']))
        story.append(Spacer(1, 0.2*72))

        # Recommendations
        story.append(Paragraph("<b>Recommendations</b>", styles['Heading2']))
        for i, rec in enumerate(data['recommendations'], 1):
            story.append(Paragraph(f"<b>{i}. {rec['action']}</b>", styles['Normal']))
            story.append(Paragraph(f"Priority: {rec['priority']}", styles['Normal']))
            story.append(Spacer(1, 0.1*72))

        doc.build(story)


## Block 4 — Knowledge Base and Explainable Engineering Logic

This section is the reasoning support layer in your notebook.

### What it contains
- a preventive maintenance knowledge base
- severity-linked recommendation lists
- a simple root-cause analysis generator
- an executive summary generator

### Why it matters
This is what makes the notebook feel like an **inspection workflow**, not just an image processor.  
The output is turned into engineering language that can support maintenance decisions.


In [ ]:

# Step 5: Knowledge Base
KNOWLEDGE_BASE = {
    'preventive_measures': [
        'Implement quarterly visual inspection protocol',
        'Maintain protective coating integrity',
        'Control environmental factors (humidity, pH)',
        'Install cathodic protection for buried structures',
        'Ensure proper drainage and ventilation',
        'Document all inspections and maintenance',
        'Train personnel on corrosion identification',
        'Use corrosion-resistant materials',
        'Apply corrosion inhibitors',
        'Establish monitoring program with trending'
    ],
    'recommendations': {
        'critical': [
            {'action': 'Immediate structural integrity assessment', 'priority': 'immediate',
             'cost': '$5,000-$15,000', 'timeline': '24-48 hours'},
            {'action': 'Emergency corrosion control measures', 'priority': 'immediate',
             'cost': '$3,000-$10,000', 'timeline': '1 week'}
        ],
        'high': [
            {'action': 'Comprehensive corrosion assessment', 'priority': 'urgent',
             'cost': '$3,000-$8,000', 'timeline': '1-2 weeks'},
            {'action': 'Surface preparation and recoating', 'priority': 'short_term',
             'cost': '$30-$80/m²', 'timeline': '1-2 months'}
        ],
        'moderate': [
            {'action': 'Schedule coating renewal', 'priority': 'short_term',
             'cost': '$25-$60/m²', 'timeline': '3-6 months'},
            {'action': 'Enhanced monitoring program', 'priority': 'short_term',
             'cost': '$2,000-$5,000/year', 'timeline': 'Ongoing'}
        ],
        'low': [
            {'action': 'Continue routine inspections', 'priority': 'maintenance',
             'cost': '$1,000-$3,000/year', 'timeline': 'Quarterly'},
            {'action': 'Monitor corrosion progression', 'priority': 'maintenance',
             'cost': '$500-$2,000/year', 'timeline': 'Ongoing'}
        ]
    }
}

def generate_rca(detections, material):
    """Generate root cause analysis"""
    types = set(d['type'] for d in detections)
    rca = []

    if 'pitting' in types:
        rca.append({
            'cause': 'Localized electrochemical cell formation (pitting corrosion)',
            'confidence': 0.84,
            'evidence': [
                'Pitting pattern indicates anodic dissolution sites',
                'Typical of chloride or sulfide contamination',
                'Passive film breakdown in localized areas'
            ]
        })

    if 'uniform' in types:
        rca.append({
            'cause': 'General atmospheric corrosion',
            'confidence': 0.81,
            'evidence': [
                'Uniform attack pattern across surface',
                f'{material} susceptible to oxidation',
                'Environmental exposure without adequate protection'
            ]
        })

    if 'steel' in material.lower():
        rca.append({
            'cause': 'Iron oxide formation on steel substrate',
            'confidence': 0.87,
            'evidence': [
                f'Material: {material}',
                'Moisture and oxygen exposure',
                'Coating absence or degradation'
            ]
        })

    return rca if rca else [{'cause': 'Environmental corrosion', 'confidence': 0.75,
                             'evidence': ['General corrosion mechanisms']}]

def generate_summary(analysis, material):
    """Generate executive summary"""
    count = len(analysis['detections'])
    area = analysis['total_affected_area']
    severity = analysis['overall_severity']

    summary = f"AI computer vision analysis identified {count} corrosion zone(s) "
    summary += f"affecting {area:.1f}% of the {material} surface. "

    if severity == 'critical':
        summary += "CRITICAL severity requires immediate intervention to prevent failure. "
    elif severity == 'high':
        summary += "HIGH severity warrants urgent remediation within 30 days. "
    elif severity == 'moderate':
        summary += "MODERATE severity suggests maintenance within 3-6 months. "
    else:
        summary += "LOW severity indicates routine monitoring is sufficient. "

    summary += "Report includes findings, root cause analysis, and preventive recommendations."
    return summary


## Block 5 — Main End-to-End Analysis Function

This is the orchestration block from your original notebook.

### What it does
- loads the uploaded image
- resizes very large images
- runs the corrosion analyzer
- builds an annotated overlay
- prepares structured report data
- generates DOCX and PDF outputs

This is effectively the **CorroLLM execution engine** in your notebook.


In [ ]:

# Step 6: Main Analysis Function
def analyze_and_generate_report(image_path, material, location=None):
    """Complete analysis pipeline"""
    print("🔍 Analyzing image...")

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Could not load image")

    # Resize if needed
    h, w = img.shape[:2]
    if max(h, w) > 2048:
        scale = 2048 / max(h, w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)))

    # Initialize analyzer
    analyzer = CorrosionAnalyzer()

    # Run analysis
    analysis = analyzer.analyze_image(img)

    print(f"✅ Found {len(analysis['detections'])} detection(s)")
    print(f"   Severity: {analysis['overall_severity'].upper()}")
    print(f"   Affected area: {analysis['total_affected_area']:.1f}%")

    # Create overlay
    overlay = analyzer.create_overlay(img, analysis['detections'])
    overlay_path = '/content/reports/overlay.jpg'
    cv2.imwrite(overlay_path, overlay)

    print("\n📸 Displaying annotated image...")
    display(IPImage(overlay_path))

    # Prepare report data
    report_data = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'material': material,
        'location': location,
        'severity': analysis['overall_severity'],
        'total_affected_area': analysis['total_affected_area'],
        'detections': analysis['detections'],
        'executive_summary': generate_summary(analysis, material),
        'rca': generate_rca(analysis['detections'], material),
        'recommendations': KNOWLEDGE_BASE['recommendations'][analysis['overall_severity']],
        'preventive_measures': KNOWLEDGE_BASE['preventive_measures']
    }

    # Generate reports
    print("\n📄 Generating reports...")
    generator = ReportGenerator()
    docx_path, pdf_path = generator.generate_reports(report_data)

    print(f"✅ DOCX report: {docx_path}")
    print(f"✅ PDF report: {pdf_path}")

    return docx_path, pdf_path, report_data


## Block 6 — Interactive Colab Run Cell

This final section matches the user-facing behavior in your original notebook.

### User flow
1. Upload one corrosion image
2. Preview the image
3. Select material type
4. Optionally add location
5. Run analysis
6. Review detections and recommendations
7. Download DOCX and PDF reports

This keeps the notebook simple for rapid inspection use in Colab.


In [ ]:

# Step 7: Interactive Interface
print("\n" + "="*70)
print("🔬 CORROLLM - AI CORROSION ANALYSIS SYSTEM")
print("="*70)
print("\n📋 Upload your corrosion image to begin analysis\n")

# Upload image
uploaded = files.upload()

if uploaded:
    # Get uploaded file
    filename = list(uploaded.keys())[0]
    filepath = f'/content/{filename}'

    print(f"\n✅ Image uploaded: {filename}")
    print("\n📸 Preview:")
    display(IPImage(filepath))

    # Get material input
    print("\n" + "="*70)
    print("MATERIAL SELECTION")
    print("="*70)
    print("1. Carbon Steel")
    print("2. Stainless Steel (304/316)")
    print("3. Aluminum Alloy")
    print("4. Galvanized Steel")
    print("5. Copper/Brass")

    material_choice = input("\nSelect material (1-5): ").strip()
    materials = {
        '1': 'Carbon Steel',
        '2': 'Stainless Steel',
        '3': 'Aluminum Alloy',
        '4': 'Galvanized Steel',
        '5': 'Copper/Brass'
    }
    material = materials.get(material_choice, 'Carbon Steel')

    location = input("Location (optional, press Enter to skip): ").strip() or None

    print("\n" + "="*70)
    print("STARTING ANALYSIS")
    print("="*70)

    try:
        # Run analysis
        docx_path, pdf_path, report_data = analyze_and_generate_report(
            filepath, material, location
        )

        # Display results
        print("\n" + "="*70)
        print("📊 ANALYSIS RESULTS")
        print("="*70)
        print(f"\n{report_data['executive_summary']}\n")

        print("🔍 DETECTIONS:")
        for i, det in enumerate(report_data['detections'], 1):
            print(f"  {i}. {det['type'].replace('_',' ').title()}")
            print(f"     Severity: {det['severity'].upper()}")
            print(f"     Confidence: {det['confidence']:.1%}")
            print(f"     Area: {det['area_percent']:.1f}%\n")

        print("💡 TOP 3 RECOMMENDATIONS:")
        for i, rec in enumerate(report_data['recommendations'][:3], 1):
            print(f"  {i}. {rec['action']}")
            print(f"     Priority: {rec['priority'].upper()}")
            print(f"     Cost: {rec['cost']}\n")

        # Download reports
        print("\n" + "="*70)
        print("📥 DOWNLOAD REPORTS")
        print("="*70)
        files.download(docx_path)
        files.download(pdf_path)

        print("\n✅ Reports downloaded successfully!")
        print("\n" + "="*70)
        print("🎉 ANALYSIS COMPLETE!")
        print("="*70)

    except Exception as e:
        print(f"\n❌ Analysis failed: {e}")
else:
    print("No file uploaded yet.")


## Notes

### What stayed the same
This notebook still uses **your original logic** for:
- contour-based corrosion candidate detection
- severity estimation
- overlay visualization
- root cause statements
- DOCX/PDF report generation

### What changed
Only the presentation changed:
- one long code cell was split into clear blocks
- each block now has an explanation
- the notebook reads more like a proper engineering workflow document

If you want, the next version can also include:
- a **workflow diagram page** at the top
- a **table of input/output for each block**
- a **journal-style pseudocode section**
